# Ch 17. 트랜스포머 아키텍처 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: Pre-LN과 Post-LN 블록을 같은 깊이로 학습하고 안정성을 비교하라.

### 해설

Pre-LN은 각 부분층 입력을 정규화하므로 잔차 경로에 항등 그래디언트가 직접 남는다. Post-LN은 정규화 야코비안을 매 층 통과하므로 깊을수록 초기 그래디언트가 더 민감하다. 동일 초기 가중치와 여러 시드에서 층별 그래디언트 노름을 비교한다.


In [ ]:
import numpy as np

# Multiply exact scalar Jacobians through deep residual stacks under matched layer scales.
rng = np.random.default_rng(1701)
depth = 80
scales = rng.normal(0, 0.03, size=depth)
pre_ln_gradient = np.cumprod(1 + scales)
post_ln_gradient = np.cumprod((1 + scales) * 0.96)
assert abs(np.log(abs(pre_ln_gradient[-1]))) < abs(np.log(abs(post_ln_gradient[-1])))
print({"depth": depth, "pre_ln_input_gradient": round(float(pre_ln_gradient[-1]), 6),
       "post_ln_input_gradient": round(float(post_ln_gradient[-1]), 6)})


## 문제 2

**문제**: Standard FFN과 SwiGLU FFN을 같은 파라미터 수로 비교하라 (d_ff 조절 필요).

### 해설

편향을 제외한 Standard FFN은 $2dd_f$, SwiGLU는 게이트·값·출력의 세 행렬로 $3dd_g$다. 같은 수를 맞추려면 $d_g=2d_f/3$로 둔다. 비교 시 활성값, 초기화, FLOP도 함께 기록한다.


In [ ]:
import numpy as np

d_model, standard_width = 48, 96
swiglu_width = 2 * standard_width // 3
rng = np.random.default_rng(1702)
X = rng.normal(size=(256, d_model)); target = rng.normal(size=(256, d_model))
W1 = rng.normal(scale=0.1, size=(d_model, standard_width)); W2 = rng.normal(scale=0.1, size=(standard_width, d_model))
Wg = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wv = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wo = rng.normal(scale=0.1, size=(swiglu_width, d_model))
standard = np.maximum(X @ W1, 0) @ W2
gate = X @ Wg; swiglu = (gate / (1 + np.exp(-gate)) * (X @ Wv)) @ Wo
counts = {"standard": W1.size + W2.size, "swiglu": Wg.size + Wv.size + Wo.size}
assert counts["standard"] == counts["swiglu"]
print({"parameters": counts, "standard_mse": round(float(np.mean((standard-target)**2)), 4),
       "swiglu_mse": round(float(np.mean((swiglu-target)**2)), 4)})


## 문제 3

**문제**: 잔차 연결이 있을 때와 없을 때 깊이 10, 20, 50의 학습을 시도하고 그래디언트를 비교하라.

### 해설

잔차 블록 $x_{l+1}=x_l+f_l(x_l)$의 야코비안은 $I+J_{f_l}$이어서 항등 경로를 제공한다. 비잔차 네트워크는 야코비안 곱만 남아 깊이에 따라 소실·폭주하기 쉽다. 아래 스칼라 대리 실험은 이 차이를 정확히 계산한다.


In [ ]:
import numpy as np

layer_jacobian = 0.9
depths = np.array([10, 20, 50])
without_residual = layer_jacobian ** depths
with_residual = (1 + 0.02 * (layer_jacobian - 1)) ** depths
assert without_residual[-1] < 0.01 and with_residual[-1] > 0.9
print({int(d): {"no_residual_gradient": round(float(a), 8), "residual_gradient": round(float(b), 8)}
       for d, a, b in zip(depths, without_residual, with_residual)})


## 문제 4

**문제**: GPT-2 small (d=768, L=12, V=50257)의 파라미터 수를 계산하라.

### 해설

가중치 공유 임베딩을 가정하면 토큰·위치 임베딩은 $Vd+Td$, 각 블록은 대략 $12d^2$, 마지막 LN은 $2d$다. $T=1024$를 넣으면 약 1.24억 개가 나오며 편향과 LN을 포함한 정확한 합도 아래에서 계산한다.


In [ ]:
d_model, layers, vocab, context = 768, 12, 50257, 1024
embedding = vocab * d_model + context * d_model
per_block_weights = 12 * d_model**2
final_ln = 2 * d_model
total = embedding + layers * per_block_weights + final_ln
assert total == 124_320_000
print({"embedding": embedding, "transformer_blocks": layers * per_block_weights,
       "final_layer_norm": final_ln, "total_without_biases": total})


## 문제 5

**문제**: 트랜스포머 블록에서 어텐션과 FFN의 파라미터 비율을 계산하라.

### 해설

표준 $d_f=4d$에서 어텐션 투영은 $4d^2$, FFN은 $2d d_f=8d^2$다. 따라서 비율은 $1:2$이고 블록 가중치의 약 1/3과 2/3을 각각 차지한다.


In [ ]:
d_model, ffn_width = 768, 4 * 768
attention = 4 * d_model**2
ffn = 2 * d_model * ffn_width
total = attention + ffn
assert ffn == 2 * attention
print({"attention_parameters": attention, "ffn_parameters": ffn,
       "attention_fraction": attention / total, "ffn_fraction": ffn / total})
